In [36]:
suppressWarnings(
  suppressPackageStartupMessages(
    {
      library(idiolect)
      library(writexl)
      library(dplyr)
      library(purrr)
      library(tibble)
    }
  )
)

In [38]:
score_grouped_performance <- function(
  completed_df,
  grouping_cols,
  scoring_cols,
  by = "case",
  progress = FALSE
) {
  purrr::map_dfr(scoring_cols, function(score_col) {
    
    completed_df %>%
      dplyr::select(dplyr::all_of(c(grouping_cols, "target", score_col))) %>%
      dplyr::rename(score = !!score_col) %>%
      dplyr::group_by(dplyr::across(dplyr::all_of(grouping_cols))) %>%
      dplyr::group_modify(function(.x, .y) {
        
        perf <- tryCatch(
          idiolect::performance(
            training = .x %>% dplyr::select(score, target),
            by = by,
            progress = progress
          ),
          error = function(e) NULL
        )
        
        if (is.null(perf) || is.null(perf$evaluation) || nrow(perf$evaluation) == 0) {
          return(tibble::tibble())
        }
        
        dplyr::bind_cols(
          tibble::tibble(scoring_col = score_col),
          perf$evaluation
        )
      }) %>%
      dplyr::ungroup()
  })
}

In [42]:
score_grouped_calibrated_performance <- function(
  completed_df,
  grouping_cols,
  scoring_cols,
  progress = FALSE
) {
  if (!"data_type" %in% names(completed_df)) {
    stop("completed_df must contain a 'data_type' column", call. = FALSE)
  }

  if (!all(c("training", "test") %in% unique(completed_df$data_type))) {
    stop("completed_df$data_type must contain both 'training' and 'test'", call. = FALSE)
  }

  match_cols <- setdiff(grouping_cols, "data_type")

  if (length(match_cols) == 0) {
    stop("After excluding 'data_type', no grouping columns remain for matching", call. = FALSE)
  }

  training_df <- completed_df %>%
    dplyr::filter(data_type == "training")

  test_df <- completed_df %>%
    dplyr::filter(data_type == "test")

  # keep only groups present in both training and test
  matched_groups <- training_df %>%
    dplyr::distinct(dplyr::across(dplyr::all_of(match_cols))) %>%
    dplyr::inner_join(
      test_df %>% dplyr::distinct(dplyr::across(dplyr::all_of(match_cols))),
      by = match_cols
    )

  message("Number of matched groups for calibration: ", nrow(matched_groups))

  purrr::map_dfr(scoring_cols, function(score_col) {

    message("Running calibrated scoring for: ", score_col)

    training_sub <- training_df %>%
      dplyr::inner_join(matched_groups, by = match_cols) %>%
      dplyr::select(dplyr::all_of(c(match_cols, "target", score_col))) %>%
      dplyr::filter(!is.na(.data[[score_col]]), !is.na(target)) %>%
      dplyr::rename(score = !!score_col)

    test_sub <- test_df %>%
      dplyr::inner_join(matched_groups, by = match_cols) %>%
      dplyr::select(dplyr::all_of(c(match_cols, "target", score_col))) %>%
      dplyr::filter(!is.na(.data[[score_col]]), !is.na(target)) %>%
      dplyr::rename(score = !!score_col)

    group_list <- matched_groups %>%
      split(seq_len(nrow(.)))

    purrr::map_dfr(group_list, function(group_row) {

      train_group <- training_sub %>%
        dplyr::semi_join(group_row, by = match_cols)

      test_group <- test_sub %>%
        dplyr::semi_join(group_row, by = match_cols)

      perf <- tryCatch(
        idiolect::performance(
          training = train_group %>% dplyr::select(score, target),
          test = test_group %>% dplyr::select(score, target),
          progress = progress
        ),
        error = function(e) {
          message(
            "Skipping calibrated group: ",
            paste(names(group_row), as.character(group_row[1, ]), sep = "=", collapse = ", "),
            ", scoring_col=", score_col,
            " | error: ", e$message
          )
          NULL
        }
      )

      if (is.null(perf) || is.null(perf$evaluation) || nrow(perf$evaluation) == 0) {
        return(tibble::tibble())
      }

      dplyr::bind_cols(
        tibble::as_tibble(group_row),
        tibble::tibble(scoring_col = score_col),
        tibble::as_tibble(perf$evaluation)
      )
    })
  })
}

In [53]:
base_loc <- "/Volumes/BCross/av_datasets_experiments/weighted_ngram_tracing_concat"

grouping_cols <- c(
  "data_type",
  "corpus",
  "scoring_model",
  "min_token_size",
  "weight",
  "alpha",
  "base"
)

scoring_cols <- c(
  "simpson_score",
  "jaccard_score"
)

## Raw Data

In [ ]:
completed_data <- readRDS(paste0(base_loc, "/completed_adj_token_size_raw.rds"))

completed_data_save_loc <- paste0(base_loc, "/idiolect_scores_raw")
calibrated_data_save_loc <- paste0(base_loc, "/calibrated_idiolect_scores_raw")

performance_df <- score_grouped_performance(
  completed_df = completed_data,
  grouping_cols = grouping_cols,
  scoring_cols = scoring_cols,
  by = "case",
  progress = FALSE
)

writexl::write_xlsx(performance_df, path = paste0(completed_data_save_loc, ".xlsx"))
print("Performance Excel File Written")
saveRDS(performance_df, file=paste0(completed_data_save_loc, ".rds"))
print("Performance RDS file written")

calibrated_df <- score_grouped_calibrated_performance(
  completed_data,
  grouping_cols,
  scoring_cols,
  progress = FALSE
)

writexl::write_xlsx(calibrated_df, path=paste0(calibrated_data_save_loc , ".xlsx"))
print("Calibrated Performance Excel File Written")
saveRDS(calibrated_df, file=paste0(calibrated_data_save_loc, ".rds"))
print("Calibrated Performance RDS file written")

print("Process complete")

## Like for Like Data

In [54]:
lfl_data <- readRDS(paste0(base_loc, "/completed_like_for_like_raw.rds"))

lfl_save_loc <- paste0(base_loc, "/lfl_idiolect_scores_raw")
calibrated_lfl_save_loc <- paste0(base_loc, "/calibrated_lfl_idiolect_scores_raw")

lfl_performance_df <- score_grouped_performance(
  completed_df = lfl_data,
  grouping_cols = grouping_cols,
  scoring_cols = scoring_cols,
  by = "case",
  progress = FALSE
)

writexl::write_xlsx(lfl_performance_df, path = paste0(lfl_save_loc, ".xlsx"))
print("Performance Excel File Written")
saveRDS(lfl_performance_df, file=paste0(lfl_save_loc, ".rds"))
print("Performance RDS file written")

lfl_calibrated_df <- score_grouped_calibrated_performance(
  lfl_data,
  grouping_cols,
  scoring_cols,
  progress = FALSE
)

writexl::write_xlsx(lfl_calibrated_df, path=paste0(calibrated_lfl_save_loc , ".xlsx"))
print("Calibrated Performance Excel File Written")
saveRDS(lfl_calibrated_df, file=paste0(calibrated_lfl_save_loc, ".rds"))
print("Calibrated Performance RDS file written")

Warning messages:
1: In predict.lm(object, newdata, se.fit, scale = 1, type = if (type ==  :
  prediction from a rank-deficient fit may be misleading
2: In predict.lm(object, newdata, se.fit, scale = 1, type = if (type ==  :
  prediction from a rank-deficient fit may be misleading
3: In predict.lm(object, newdata, se.fit, scale = 1, type = if (type ==  :
  prediction from a rank-deficient fit may be misleading
4: In predict.lm(object, newdata, se.fit, scale = 1, type = if (type ==  :
  prediction from a rank-deficient fit may be misleading
5: In predict.lm(object, newdata, se.fit, scale = 1, type = if (type ==  :
  prediction from a rank-deficient fit may be misleading
6: In predict.lm(object, newdata, se.fit, scale = 1, type = if (type ==  :
  prediction from a rank-deficient fit may be misleading


[1] "Performance Excel File Written"
[1] "Performance RDS file written"
Number of matched groups for calibration: 255
Running calibrated scoring for: simpson_score
Skipping calibrated group: corpus=Amazon, scoring_model=gpt2, min_token_size=9, weight=exp, alpha=NaN, base=2, scoring_col=simpson_score | error: Argument mu must be a nonempty numeric vector
Skipping calibrated group: corpus=Amazon, scoring_model=gpt2, min_token_size=10, weight=exp, alpha=NaN, base=2, scoring_col=simpson_score | error: Argument mu must be a nonempty numeric vector
Skipping calibrated group: corpus=Amazon, scoring_model=gpt2, min_token_size=11, weight=exp, alpha=NaN, base=2, scoring_col=simpson_score | error: Argument mu must be a nonempty numeric vector
Skipping calibrated group: corpus=Amazon, scoring_model=gpt2, min_token_size=12, weight=exp, alpha=NaN, base=2, scoring_col=simpson_score | error: Argument mu must be a nonempty numeric vector
Skipping calibrated group: corpus=Amazon, scoring_model=gpt2, min